# Phase 2: stage the billing address backfill

Freezes the population (`BillingStreet` set AND the entire `PersonMailing*` destination
empty) into `crm_imp_person_accounts`, one row per account, `_operation='update'`.
Source values are the Billing side: `address=BillingStreet`, `city=BillingCity`,
`postal_code=BillingPostalCode`, `country=BillingCountryCode__c` (both sides ISO-2,
see `02_recon_addresses.sql`).

**This notebook only writes to the local MySQL staging table. It never touches Salesforce.**

Decisions (Arsal 2026-08-25): copy only, Billing* stays untouched; accounts with a
partially filled PersonMailing side (~505) are skipped entirely — only process where
the destination is empty.

Prerequisites: mirrors refreshed (`camping-grubhof-import/refresh_sf_mirrors.py`),
then `01_create_mirror_indexes.sql` re-applied (the refresh drops the indexes).
Recon numbers (2026-08-25 mirror): expected population ~732,363.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root: config, mysql_client

from config import load_mysql_config
from mysql_client import MySQLClient

BATCH_ID = "2026-08-25_billing_backfill"

db = MySQLClient(load_mysql_config())
print("connected | batch:", BATCH_ID)

## 1. Mirror freshness

`MAX(LastModifiedDate)` should be within a day of the refresh; if it is stale, stop and
re-run the refresh before staging — the live integration writes PersonMailing* all day,
and a stale mirror stages accounts whose destination is no longer empty. (The loader
has a live skip-check as the last line of defense, but staging fresh keeps the numbers
honest.)

In [ ]:
row = db.fetch_one(
    "SELECT COUNT(*) AS n, MAX(LastModifiedDate) AS newest FROM crm_person_account_sfid_prod"
)
print(f"crm_person_account_sfid_prod  {row['n']:>12,}  newest: {row['newest']}")

## 2. Population counts

Expected from the 2026-08-25 recon: ~732,363 accounts (732,868 billing-only minus
~505 with a partially filled PersonMailing side). Field completeness inside the
population: city ~99.6%, postal ~98.5%, country ~99.9%. Small drift after a refresh
is possible — investigate anything larger than a few hundred.

In [ ]:
POPULATION_WHERE = """
    a.BillingStreet IS NOT NULL AND a.BillingStreet <> ''
    AND (a.PersonMailingStreet     IS NULL OR a.PersonMailingStreet = '')
    AND (a.PersonMailingCity       IS NULL OR a.PersonMailingCity = '')
    AND (a.PersonMailingPostalCode IS NULL OR a.PersonMailingPostalCode = '')
    AND (a.PersonMailingCountry    IS NULL OR a.PersonMailingCountry = '')
"""

counts = db.fetch_one(f"""
    SELECT COUNT(*) AS accounts,
           SUM(a.BillingCity IS NOT NULL AND a.BillingCity <> '') AS has_city,
           SUM(a.BillingPostalCode IS NOT NULL AND a.BillingPostalCode <> '') AS has_postal,
           SUM(a.BillingCountryCode__c IS NOT NULL AND a.BillingCountryCode__c <> '') AS has_country
    FROM crm_person_account_sfid_prod a
    WHERE {POPULATION_WHERE}
""")
expected_accounts = int(counts["accounts"])
print(f"population accounts: {expected_accounts:,}")
print(f"  with city:    {int(counts['has_city']):,}")
print(f"  with postal:  {int(counts['has_postal']):,}")
print(f"  with country: {int(counts['has_country']):,}")

## 3. Stage the batch

One row per account. Guarded: refuses to run if the batch id already has rows — rerun
after a mistake means deleting the batch first, deliberately, not re-executing the cell.

Also adds the `_billing_processed_at` bookkeeping column if the table does not have it
yet (the generic `_processed_at` is already claimed by the older update scripts).

In [ ]:
col = db.fetch_one("""
    SELECT COUNT(*) AS n FROM information_schema.columns
    WHERE table_schema = DATABASE()
      AND table_name = 'crm_imp_person_accounts'
      AND column_name = '_billing_processed_at'
""")["n"]
if not col:
    db.execute("ALTER TABLE crm_imp_person_accounts ADD COLUMN _billing_processed_at DATETIME NULL")
    print("column _billing_processed_at added")
else:
    print("column _billing_processed_at exists")

existing = db.fetch_one(
    "SELECT COUNT(*) AS n FROM crm_imp_person_accounts WHERE _batch_id = %s",
    (BATCH_ID,),
)["n"]
assert existing == 0, f"{existing} rows already staged under this batch id — not re-inserting"

inserted = db.execute(f"""
    INSERT INTO crm_imp_person_accounts
        (_operation, _batch_id, _excluded, source, last_name, email,
         sf_account_id, sf_person_contact_id,
         address, city, postal_code, country)
    SELECT
        'update', %s, 0, 'billing_backfill',
        a.LastName, a.PersonEmail,
        a.Id, a.PersonContactId,
        a.BillingStreet, NULLIF(a.BillingCity, ''), NULLIF(a.BillingPostalCode, ''),
        NULLIF(a.BillingCountryCode__c, '')
    FROM crm_person_account_sfid_prod a
    WHERE {POPULATION_WHERE}
""", (BATCH_ID,))

print(f"staged {inserted:,} rows under {BATCH_ID}")
assert inserted == expected_accounts, f"staged {inserted} but section 2 predicted {expected_accounts}" 

## 4. Verification and predicted after-state

These numbers are the load contract: the dry-run must produce exactly this many rows,
and after the load the live count of "BillingStreet set, PersonMailingStreet empty"
should be ~0 plus whatever the live skip-check left out.

In [ ]:
summary = db.fetch_one("""
    SELECT COUNT(*) AS rows_staged,
           COUNT(DISTINCT sf_account_id) AS accounts,
           SUM(address IS NULL OR address = '') AS empty_street,
           SUM(city IS NOT NULL) AS with_city,
           SUM(postal_code IS NOT NULL) AS with_postal,
           SUM(country IS NOT NULL) AS with_country
    FROM crm_imp_person_accounts
    WHERE _batch_id = %s
""", (BATCH_ID,))
for k, v in summary.items():
    print(f"{k:15s} {int(v):,}")

assert int(summary["rows_staged"]) == int(summary["accounts"]), "duplicate sf_account_id staged"
assert int(summary["empty_street"]) == 0, "rows without a street value staged"
print("\none row per account, every row has a street — staging frozen")

## 5. Review export (no writes): both sides filled and different

The 961+6 accounts where Billing* and PersonMailing* are both filled but differ are
NOT part of the backfill (the Mailing side is presumed correct). Exported for a human
review; PII, lands in the gitignored `local_data/`.

In [ ]:
review = db.fetch_df("""
    SELECT Id, PersonEmail, BillingStreet, PersonMailingStreet, CreatedDate, SourceSystem__pc
    FROM crm_person_account_sfid_prod
    WHERE BillingStreet IS NOT NULL AND BillingStreet <> ''
      AND PersonMailingStreet IS NOT NULL AND PersonMailingStreet <> ''
      AND BillingStreet <> PersonMailingStreet
""")
out = Path.cwd().parent / "local_data" / "billing_mailing_differ_review.csv"
out.parent.mkdir(parents=True, exist_ok=True)
review.to_csv(out, index=False)
print(f"{len(review):,} differing accounts exported to {out}")

## Next

- Dry-run: `python billing-address-backfill/update_mailing_addresses.py <batch_id> --dry-run`
  (or via `04_run_billing_backfill.ipynb`, which drives all of phases 3-6).
- Probe: one account in prod, UI eyeball (04, `RUN_PROBE` gate).
- Bulk load: **does not run without Arsal's explicit go-ahead** (04, `RUN_LOAD` gate).